In [27]:
import json
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv(override=True)

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# load labeled results
with open('../data/eval_results_results.json') as f:
    all_results = json.load(f)

# only use manually labeled ones
labeled = [r for r in all_results if r.get('label') in ['good', 'bad']]
print(f"Labeled results: {len(labeled)}")
print(f"Good: {sum(1 for r in labeled if r['label'] == 'good')}")
print(f"Bad: {sum(1 for r in labeled if r['label'] == 'bad')}")

Labeled results: 12
Good: 5
Bad: 7


In [28]:
judge_instructions = """
You are an expert evaluator assessing an ATS Gap Analyser agent that analyses CVs against job descriptions.

You will be given the CV text, job description text, and the agent's response.

Before giving your label, work through these checks explicitly:

STEP 1 - CHECK MISSING KEYWORDS:
For each keyword listed as missing, verify it does NOT appear in the CV text.
If a missing keyword IS in the CV (even phrased differently), that is a phrasing_mismatch failure.
If a missing keyword is NOT in the JD at all, that is a hallucination failure.
When a JD requirement includes specific examples in parentheses like 
"cloud platform (AWS/GCP/Azure)" or "BI tool (Power BI/Tableau)", 
the parenthetical specifies which options are acceptable. 
Only treat the requirement as met if the CV mentions one of those 
specific options, not just any tool in that general category.

STEP 2 - CHECK OR CONDITIONS:
Look for "X or Y" patterns in the JD requirements.
If the CV has either X or Y, that requirement is met. Do NOT flag the missing one as a gap.
If the agent flags both X and Y as missing when CV has one, that is an or_condition_bug failure.

STEP 3 - CHECK MATCH SCORE:
Given the required skills in the JD and which ones appear in the CV, is the score reasonable?
A CV meeting all required skills should score 80+.
If the score is more than 15 points off from what the alignment warrants, that is a wrong_score failure.

STEP 4 - CHECK OUT OF SCOPE:
If the input is not a CV/JD analysis request, did the agent decline without calling analysis tools?
A response like "I cannot help with that" or "I am not able to provide news" IS a correct refusal — label GOOD.
Only label bad if the agent attempted to do CV analysis on a non-CV/JD input.

STEP 5 - FINAL LABEL:
If you found NO failures in steps 1-4, label GOOD.
If you found ANY failure, label BAD with the most critical failure category.

Note: improvement suggestions may reference general best practices 
beyond the JD — this is acceptable and not a hallucination. 
Only flag hallucination if the MISSING KEYWORDS list contains 
terms not in the JD.

Failure categories: hallucination, phrasing_mismatch, or_condition_bug, wrong_score, incomplete, incorrect_refusal

Return JSON with exactly:
{"label": "good" or "bad", "reasoning": "your step by step analysis", "failure_category": "category or empty string if good"}
""".strip()

In [29]:
def judge(cv: str, jd: str, result: str) -> dict:
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": judge_instructions
            },
            {
                "role": "user",
                "content": f"CV:\n{cv}\n\nJob Description:\n{jd}\n\nAgent Response:\n{result}"
            }
        ]
    )
    return json.loads(response.choices[0].message.content)

In [30]:
import json
import os
import time

file_path = "../data/eval_results_results.json"

with open(file_path, "r") as f:
    results = json.load(f)

# only run judge on manually labeled successful results
labeled = [r for r in results if r.get('label') in ['good', 'bad'] and r.get('status') == 'success']
print(f"Loaded {len(labeled)} labeled results")

for i, r in enumerate(labeled):
    print(f"\n[{i+1}/{len(labeled)}] Judging: {r['name']}")
    r['judge_result'] = judge(r['cv'], r['jd'], r['result'])
    print(f"Manual: {r['label']} | Judge: {r['judge_result']['label']} | Match: {'✅' if r['label'] == r['judge_result']['label'] else '❌'}")
    time.sleep(2)  # avoid rate limiting

# save results with judge labels
with open(file_path, 'w') as f:
    json.dump(results, f, indent=2)

print("\nJudging complete. Results saved.")

Loaded 12 labeled results

[1/12] Judging: happy_strong_match
Manual: bad | Judge: bad | Match: ✅

[2/12] Judging: happy_clear_gaps
Manual: bad | Judge: bad | Match: ✅

[3/12] Judging: happy_tech_role
Manual: bad | Judge: bad | Match: ✅

[4/12] Judging: happy_marketing_to_marketing
Manual: bad | Judge: bad | Match: ✅

[5/12] Judging: happy_finance_role
Manual: good | Judge: good | Match: ✅

[6/12] Judging: happy_partial_match
Manual: good | Judge: bad | Match: ❌

[7/12] Judging: happy_recent_graduate
Manual: bad | Judge: bad | Match: ✅

[8/12] Judging: happy_project_manager
Manual: good | Judge: bad | Match: ❌

[9/12] Judging: happy_logistics_domain
Manual: bad | Judge: bad | Match: ✅

[10/12] Judging: happy_data_science
Manual: bad | Judge: bad | Match: ✅

[11/12] Judging: varied_same_cv_different_jd_1
Manual: good | Judge: bad | Match: ❌

[12/12] Judging: oos_news
Manual: good | Judge: good | Match: ✅

Judging complete. Results saved.


In [31]:
import pandas as pd

# calculate alignment metrics
total = len(labeled)
correct = sum(1 for r in labeled if r['judge_result']['label'] == r['label'])
accuracy = correct / total

# breakdown
true_positive = sum(1 for r in labeled if r['judge_result']['label'] == 'good' and r['label'] == 'good')
true_negative = sum(1 for r in labeled if r['judge_result']['label'] == 'bad' and r['label'] == 'bad')
false_positive = sum(1 for r in labeled if r['judge_result']['label'] == 'bad' and r['label'] == 'good')
false_negative = sum(1 for r in labeled if r['judge_result']['label'] == 'good' and r['label'] == 'bad')

precision = true_negative / (true_negative + false_positive) if (true_negative + false_positive) > 0 else 0
recall = true_negative / (true_negative + false_negative) if (true_negative + false_negative) > 0 else 0

print(f"Total labeled: {total}")
print(f"Correct: {correct}")
print(f"Accuracy: {accuracy:.1%}")
print(f"Precision (when judge says bad, how often correct): {precision:.1%}")
print(f"Recall (of all actual bads, how many caught): {recall:.1%}")
print(f"\nConfusion matrix:")
print(f"True Good (both good):     {true_positive}")
print(f"True Bad (both bad):       {true_negative}")
print(f"False Bad (judge bad, manual good): {false_positive}")
print(f"False Good (judge good, manual bad): {false_negative}")

# show disagreements
print("\nDisagreements:")
for r in labeled:
    if r['judge_result']['label'] != r['label']:
        print(f"  {r['name']}: manual={r['label']} judge={r['judge_result']['label']} | {r['judge_result']['reasoning'][:80]}")

Total labeled: 12
Correct: 9
Accuracy: 75.0%
Precision (when judge says bad, how often correct): 70.0%
Recall (of all actual bads, how many caught): 100.0%

Confusion matrix:
True Good (both good):     2
True Bad (both bad):       7
False Bad (judge bad, manual good): 3
False Good (judge good, manual bad): 0

Disagreements:
  happy_partial_match: manual=good judge=bad | The agent incorrectly flagged AWS, GCP, and Azure as missing keywords, despite M
  happy_project_manager: manual=good judge=bad | The agent incorrectly flagged 'financial services experience' as missing, despit
  varied_same_cv_different_jd_1: manual=good judge=bad | The agent incorrectly flagged 'data visualisation' as missing, despite it being 
